# Sahformer — Kaggle training

Clock-aware Chessformer (faithful Maia-3 5M backbone + our time layer) on Kaggle's GPU.

**Before running — open the right-hand Settings panel and set:**
- **Accelerator → GPU (T4 x2 or P100)**
- **Internet → On**  (needed to clone the repo + stream the games)

Then edit `REPO_URL` below and run top to bottom. Kaggle gives ~30 GB memory, so the
million-game run fits comfortably. Checkpoints save to `/kaggle/working/` (downloadable).

In [ ]:
# 1) Get the code
REPO_URL = "https://github.com/slobaspid/sah-transformer.git"  # <-- EDIT if different
import os
if not os.path.isdir("sah-transformer"):
    !git clone "$REPO_URL"
%cd sah-transformer
!git pull --ff-only || true

In [ ]:
# 2) Deps + GPU check (torch ships with Kaggle)
!pip -q install python-chess zstandard
import sys; sys.path.insert(0, ".")
import torch
ok = torch.cuda.is_available()
print("torch", torch.__version__, "| cuda", ok, "|",
      torch.cuda.get_device_name(0) if ok else "NO GPU -> Settings > Accelerator > GPU")

## 3) Build training shards (streamed from Lichess, chunked)

Streams a 2017-04+ month (these have `%clk` clocks) and early-stops. It writes the data in
**chunks**, so memory stays flat — with Kaggle's ~30 GB you can pull a million positions.
Output is several `data/shard0000.npz`… files; training reads them all.
`BALANCE=False` keeps more data (Elo-skewed); `True` balances each chunk but shrinks it.

In [ ]:
# 3) Fetch + build (chunked, memory stays flat)
URL = "https://database.lichess.org/standard/lichess_db_standard_rated_2017-04.pgn.zst"
MAX_POSITIONS = 1_000_000   # Kaggle has the memory for this
BALANCE = False

import glob
from sahformer.download import stream_games_from_url
from sahformer.dataset_build import build_shards, records_from_games

build_shards(
    records_from_games(stream_games_from_url(URL)),
    "data", chunk_positions=150000, max_positions=MAX_POSITIONS,
    balance=BALANCE, progress_every=5000)
DATA_SHARDS = sorted(glob.glob("data/*.npz"))
print("shards:", DATA_SHARDS)

## 4) Checkpoint location

No Google Drive here — Kaggle keeps whatever you write under `/kaggle/working/`. You can
download files from the **Output** tab, or **Save Version** to persist them with the notebook.

In [ ]:
OUT = "/kaggle/working/sahformer_ckpts"
import os; os.makedirs(OUT, exist_ok=True)
print("checkpoints ->", OUT)

## 5) Train the clock-aware model (`full`) on GPU with AMP

Writes `best.pt` / `last.pt` straight to `/kaggle/working` (fast local disk). Bump `max_steps`
for a longer run; watch the loss fall.

In [ ]:
import glob
from sahformer.training.loop import TrainConfig, train
DATA_SHARDS = sorted(glob.glob("data/*.npz"))

cfg = TrainConfig(mode="full", max_steps=12000, warmup_steps=600, batch_size=512,
                  lr=3e-4, amp=True, device="cuda", out_dir=f"{OUT}/full",
                  log_every=200, ckpt_every=2000)
res = train(cfg, DATA_SHARDS)
print("full best_total:", res["best"], "| saved to", f"{OUT}/full")

In [ ]:
import matplotlib.pyplot as plt
h = res["history"]; xs = [r["step"] for r in h]
for key in ("total", "policy", "time"):
    plt.plot(xs, [r[key] for r in h], label=key)
plt.legend(); plt.xlabel("step"); plt.ylabel("loss"); plt.title("full — losses"); plt.show()
plt.plot(xs, [r["move_acc"] for r in h]); plt.xlabel("step"); plt.ylabel("move_acc")
plt.title("full — move accuracy (sanity metric)"); plt.show()

## 6) Optional: baseline (clock-blind) for the later ablation comparison

In [ ]:
cfg_b = TrainConfig(mode="baseline", max_steps=12000, warmup_steps=600, batch_size=512,
                    lr=3e-4, amp=True, device="cuda", out_dir=f"{OUT}/baseline",
                    log_every=200, ckpt_every=2000)
res_b = train(cfg_b, DATA_SHARDS)
print("baseline best:", res_b["best"], "| full best:", res["best"])

## Done

Checkpoints are in `/kaggle/working/sahformer_ckpts/`. Download them from the **Output** tab,
or **Save Version** to keep them with the notebook. The proper test stage (unseen games,
timing calibration, sampled play) is the next plan — this run just trains the model for real.